# Multi-Agent Debate System

This notebook builds a simple debate workflow with:
- a curator-provided topic
- one Groq debater
- one Gemini debater
- one judge model that scores the debate


## 1. Environment Setup

Create a `.env` file with your API keys before running the notebook.

Example:
```env
GROQ_API_KEY=your_groq_key_here
GOOGLE_API_KEY=your_google_key_here
```

In [9]:
import json
import os
from textwrap import dedent

from dotenv import load_dotenv
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    raise ValueError("Missing GROQ_API_KEY in environment or .env file.")

if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("Missing GOOGLE_API_KEY in environment or .env file.")

print("Environment variables loaded successfully.")

Environment variables loaded successfully.


In [10]:
# You can change these model names if you want to try other supported models.
GROQ_MODEL = "llama-3.3-70b-versatile"
GEMINI_MODEL = "gemini-2.0-flash"
JUDGE_MODEL_PROVIDER = "gemini"  # choose from: "groq", "gemini"
ENABLE_GROQ_FALLBACK = True

groq_llm = ChatGroq(model=GROQ_MODEL, temperature=0.3)
gemini_llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0.3)

judge_llm = gemini_llm if JUDGE_MODEL_PROVIDER == "gemini" else ChatGroq(model=GROQ_MODEL, temperature=0)

print(f"Groq debater model: {GROQ_MODEL}")
print(f"Gemini debater model: {GEMINI_MODEL}")
print(f"Judge provider: {JUDGE_MODEL_PROVIDER}")
print(f"Groq fallback enabled: {ENABLE_GROQ_FALLBACK}")

Groq debater model: llama-3.3-70b-versatile
Gemini debater model: gemini-2.0-flash
Judge provider: gemini
Groq fallback enabled: True


In [11]:
debate_config = {
    "topic": "Should AI tutors be used as a primary learning tool in schools?",
    "agent_a_name": "Groq Debater",
    "agent_b_name": "Gemini Debater",
    "agent_a_stance": "Pro",
    "agent_b_stance": "Con",
    "tone": "professional, evidence-aware, respectful",
    "opening_word_limit": 180,
    "rebuttal_word_limit": 160,
    "closing_word_limit": 120,
    "judging_criteria": [
        "logical strength",
        "clarity",
        "responsiveness to the opponent",
        "practical insight"
    ]
}

debate_config

{'topic': 'Should AI tutors be used as a primary learning tool in schools?',
 'agent_a_name': 'Groq Debater',
 'agent_b_name': 'Gemini Debater',
 'agent_a_stance': 'Pro',
 'agent_b_stance': 'Con',
 'tone': 'professional, evidence-aware, respectful',
 'opening_word_limit': 180,
 'rebuttal_word_limit': 160,
 'closing_word_limit': 120,
 'judging_criteria': ['logical strength',
  'clarity',
  'responsiveness to the opponent',
  'practical insight']}

## 2. Debate Helpers

In [12]:
def format_transcript(transcript):
    if not transcript:
        return "No prior debate transcript yet."

    lines = []
    for item in transcript:
        lines.append(f"[{item['round'].upper()}] {item['speaker']} ({item['stance']}):\n{item['message']}")
    return "\n\n".join(lines)


def build_debater_chain(llm):
    prompt = ChatPromptTemplate.from_template(
        dedent(
            """
            You are {agent_name}, a debate participant.
            Your stance is: {stance}
            Topic: {topic}
            Tone: {tone}
            Current round: {round_name}
            Max words: {word_limit}

            Debate transcript so far:
            {transcript}

            Instructions:
            - Argue only for your assigned stance.
            - Be concise but persuasive.
            - Directly address the other side when relevant.
            - Do not invent citations.
            - Return only the debate text for this round.
            """
        )
    )
    return prompt | llm


def run_debater(chain, *, agent_name, stance, config, transcript, round_name, word_limit):
    response = chain.invoke(
        {
            "agent_name": agent_name,
            "stance": stance,
            "topic": config["topic"],
            "tone": config["tone"],
            "round_name": round_name,
            "word_limit": word_limit,
            "transcript": format_transcript(transcript),
        }
    )
    return response.content if hasattr(response, "content") else str(response)


def invoke_with_fallback(primary_chain, fallback_chain, *, agent_name, stance, config, transcript, round_name, word_limit):
    try:
        return run_debater(
            primary_chain,
            agent_name=agent_name,
            stance=stance,
            config=config,
            transcript=transcript,
            round_name=round_name,
            word_limit=word_limit,
        )
    except Exception as exc:
        message = str(exc)
        quota_related = "RESOURCE_EXHAUSTED" in message or "quota" in message.lower() or "429" in message
        if ENABLE_GROQ_FALLBACK and fallback_chain is not None and quota_related:
            print(f"Fallback triggered for {agent_name} in {round_name}: using Groq because the primary model is unavailable.")
            return run_debater(
                fallback_chain,
                agent_name=agent_name,
                stance=stance,
                config=config,
                transcript=transcript,
                round_name=round_name,
                word_limit=word_limit,
            )
        raise


def build_judge_chain(llm):
    parser = JsonOutputParser()
    prompt = ChatPromptTemplate.from_template(
        dedent(
            """
            You are an impartial debate judge.

            Topic: {topic}
            Judging criteria: {judging_criteria}

            Debate transcript:
            {transcript}

            Return valid JSON with this schema:
            {{
              "winner": "<speaker name>",
              "scorecard": {{
                "<speaker name>": {{
                  "logical_strength": <1-10>,
                  "clarity": <1-10>,
                  "responsiveness": <1-10>,
                  "practical_insight": <1-10>
                }}
              }},
              "reasoning": "short explanation",
              "improvement_tips": {{
                "<speaker name>": "one short suggestion"
              }}
            }}

            Return JSON only.
            """
        )
    )
    return prompt | llm | parser


groq_debater_chain = build_debater_chain(groq_llm)
gemini_debater_chain = build_debater_chain(gemini_llm)
judge_chain = build_judge_chain(judge_llm)

## 3. Debate Orchestration

In [13]:
def add_turn(transcript, round_name, speaker, stance, message):
    transcript.append(
        {
            "round": round_name,
            "speaker": speaker,
            "stance": stance,
            "message": message.strip(),
        }
    )


def run_debate(config):
    transcript = []

    agent_a_opening = run_debater(
        groq_debater_chain,
        agent_name=config["agent_a_name"],
        stance=config["agent_a_stance"],
        config=config,
        transcript=transcript,
        round_name="opening",
        word_limit=config["opening_word_limit"],
    )
    add_turn(transcript, "opening", config["agent_a_name"], config["agent_a_stance"], agent_a_opening)

    agent_b_opening = invoke_with_fallback(
        gemini_debater_chain,
        groq_debater_chain,
        agent_name=config["agent_b_name"],
        stance=config["agent_b_stance"],
        config=config,
        transcript=transcript,
        round_name="opening",
        word_limit=config["opening_word_limit"],
    )
    add_turn(transcript, "opening", config["agent_b_name"], config["agent_b_stance"], agent_b_opening)

    agent_a_rebuttal = run_debater(
        groq_debater_chain,
        agent_name=config["agent_a_name"],
        stance=config["agent_a_stance"],
        config=config,
        transcript=transcript,
        round_name="rebuttal",
        word_limit=config["rebuttal_word_limit"],
    )
    add_turn(transcript, "rebuttal", config["agent_a_name"], config["agent_a_stance"], agent_a_rebuttal)

    agent_b_rebuttal = invoke_with_fallback(
        gemini_debater_chain,
        groq_debater_chain,
        agent_name=config["agent_b_name"],
        stance=config["agent_b_stance"],
        config=config,
        transcript=transcript,
        round_name="rebuttal",
        word_limit=config["rebuttal_word_limit"],
    )
    add_turn(transcript, "rebuttal", config["agent_b_name"], config["agent_b_stance"], agent_b_rebuttal)

    agent_a_closing = run_debater(
        groq_debater_chain,
        agent_name=config["agent_a_name"],
        stance=config["agent_a_stance"],
        config=config,
        transcript=transcript,
        round_name="closing",
        word_limit=config["closing_word_limit"],
    )
    add_turn(transcript, "closing", config["agent_a_name"], config["agent_a_stance"], agent_a_closing)

    agent_b_closing = invoke_with_fallback(
        gemini_debater_chain,
        groq_debater_chain,
        agent_name=config["agent_b_name"],
        stance=config["agent_b_stance"],
        config=config,
        transcript=transcript,
        round_name="closing",
        word_limit=config["closing_word_limit"],
    )
    add_turn(transcript, "closing", config["agent_b_name"], config["agent_b_stance"], agent_b_closing)

    try:
        verdict = judge_chain.invoke(
            {
                "topic": config["topic"],
                "judging_criteria": ", ".join(config["judging_criteria"]),
                "transcript": format_transcript(transcript),
            }
        )
    except Exception as exc:
        message = str(exc)
        quota_related = "RESOURCE_EXHAUSTED" in message or "quota" in message.lower() or "429" in message
        if ENABLE_GROQ_FALLBACK and JUDGE_MODEL_PROVIDER == "gemini" and quota_related:
            print("Fallback triggered for judge: using Groq because Gemini is unavailable.")
            fallback_judge_chain = build_judge_chain(ChatGroq(model=GROQ_MODEL, temperature=0))
            verdict = fallback_judge_chain.invoke(
                {
                    "topic": config["topic"],
                    "judging_criteria": ", ".join(config["judging_criteria"]),
                    "transcript": format_transcript(transcript),
                }
            )
        else:
            raise

    return transcript, verdict

## 4. Run the Debate

In [14]:
transcript, verdict = run_debate(debate_config)

print("Debate complete. Total turns:", len(transcript))

Fallback triggered for Gemini Debater in opening: using Groq because the primary model is unavailable.
Fallback triggered for Gemini Debater in rebuttal: using Groq because the primary model is unavailable.
Fallback triggered for Gemini Debater in closing: using Groq because the primary model is unavailable.
Fallback triggered for judge: using Groq because Gemini is unavailable.
Debate complete. Total turns: 6


In [15]:
for turn in transcript:
    print(f"\n{'=' * 80}")
    print(f"Round: {turn['round'].title()}")
    print(f"Speaker: {turn['speaker']} ({turn['stance']})")
    print(turn['message'])


Round: Opening
Speaker: Groq Debater (Pro)
Honorable opponent, I'm pleased to argue in favor of AI tutors as a primary learning tool in schools. AI-powered systems have shown significant potential in personalizing education, catering to individual learning styles and pace. They can also provide real-time feedback, freeing human instructors to focus on more complex and creative aspects of teaching. I acknowledge concerns about AI's limitations, but I believe their benefits outweigh the drawbacks. I look forward to addressing your concerns and presenting evidence to support the effective integration of AI tutors in our education system.

Round: Opening
Speaker: Gemini Debater (Con)
While I appreciate my opponent's enthusiasm for AI tutors, I must express my concerns about their limitations as a primary learning tool. Groq Debater highlights personalization and real-time feedback as key benefits, but I argue that these advantages are outweighed by the lack of human interaction, empathy, 

In [16]:
print(json.dumps(verdict, indent=2))

{
  "winner": "Gemini Debater",
  "scorecard": {
    "Groq Debater": {
      "logical_strength": 7,
      "clarity": 8,
      "responsiveness": 9,
      "practical_insight": 6
    },
    "Gemini Debater": {
      "logical_strength": 8,
      "clarity": 9,
      "responsiveness": 9,
      "practical_insight": 8
    }
  },
  "reasoning": "Gemini Debater presented a more compelling argument by effectively highlighting the importance of human interaction, empathy, and critical thinking skills in education, which AI tutors currently cannot replicate.",
  "improvement_tips": {
    "Groq Debater": "Provide more concrete evidence to address concerns about AI tutors' limitations in understanding nuances and emotional intelligence.",
    "Gemini Debater": "Consider acknowledging potential benefits of AI tutors in specific contexts, such as supplemental learning tools, to strengthen the argument."
  }
}


## 5. Next Ideas

After the MVP works, you can extend it with:
- multiple rebuttal rounds
- cross-examination questions
- retrieval for evidence-backed arguments
- user scoring alongside the judge's scoring
- LangGraph state management for more complex control flow
